[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wdittaya/AI-for-digital-health-course/blob/main/lab09_rag_and_agents.ipynb)

# Lab 9: RAG และเอเจนต์ปัญญาประดิษฐ์ (AI Agent) สำหรับงานสุขภาพ

**รายวิชา:** 3099704 AI for Digital Health (2026) · **คาบเรียน:** วันเสาร์ที่ 17 ตุลาคม 2026 (ภาคบ่าย) · **เวลาโดยประมาณ:** 90 นาที

## วัตถุประสงค์การเรียนรู้

- สร้างระบบ RAG (Retrieval-Augmented Generation) บนคลังความรู้ทางการแพทย์ขนาดเล็ก
- ใช้ embedding แบบโอเพนซอร์ส (open source) และฐานข้อมูลเวกเตอร์ (vector store) ซึ่งทำงานบน Colab ได้โดยไม่มีค่าใช้จ่าย
- ให้แบบจำลองภาษาขนาดใหญ่ (large language model: LLM) ตอบคำถามโดยอ้างอิงเอกสารที่ค้นคืนได้ (grounding) เพื่อลดการสร้างข้อมูลที่ไม่มีอยู่จริง (hallucination)
- กำหนดให้ระบบตอบว่า *"Not in the guidelines"* (ไม่มีในแนวทางเวชปฏิบัติ) แทนการคาดเดา
- อธิบายการพัฒนาจาก RAG ไปสู่เอเจนต์ปัญญาประดิษฐ์ที่เรียกใช้เครื่องมือ (tool) ได้

> **แนวทางการทำปฏิบัติการ** (งานเดี่ยว)
>
> - ✅ **ส่วนหลัก — สำหรับผู้เรียนทุกคน ไม่จำเป็นต้องมีพื้นฐานการเขียนโปรแกรม** ให้รันเซลล์ (cell) ตามลำดับจากบนลงล่าง (คลิกปุ่ม ▶ หรือกด **Shift+Enter**) ⭐ **Exercise** แต่ละข้อกำหนดให้เปลี่ยนค่าหรือเติมช่องว่าง (`...`) เพียงจุดเดียว ถัดจากแบบฝึกหัดทุกข้อมีเซลล์ ✅ **เฉลย** ผู้เรียนควรทำแบบฝึกหัดด้วยตนเองก่อน หากไม่สามารถดำเนินการต่อได้ ให้รันเซลล์เฉลยแล้วดำเนินการในขั้นถัดไป จากนั้นให้ตอบคำถาม ✍️ ในเซลล์ข้อความ ซึ่งถือเป็น**ชิ้นงานที่ต้องส่ง** หากพบปัญหา สามารถสอบถามผู้สอนผ่านช่องแชตของ Zoom
> - 🏆 **โจทย์ท้าทายบนตารางอันดับ — ไม่บังคับ สำหรับผู้เรียนที่เขียนโปรแกรมได้** อยู่ท้ายโน้ตบุ๊ก (notebook) และ*ไม่นับ*เป็นส่วนหนึ่งของชิ้นงานที่ต้องส่ง

> **Runtime (แนะนำให้ใช้ GPU):** แบบจำลองภาษาทำงานบน GPU ได้เร็วกว่ามาก ให้เลือก `Runtime → Change runtime type → T4 GPU → Save` ใน Colab *ก่อน*รันเซลล์แรก หากไม่มี GPU ปฏิบัติการนี้ยังดำเนินการได้ แต่ LLM จะใช้เวลาตอบประมาณ 30–90 วินาทีต่อคำถาม แทนที่จะใช้เวลาเพียงไม่กี่วินาที

> ทุกขั้นตอนทำงานบน [Google Colab](https://colab.research.google.com) รุ่นที่ไม่มีค่าใช้จ่าย โดยไม่ต้องสมัครสมาชิกและไม่ต้องใช้ API key คลังความรู้ในปฏิบัติการนี้เป็น**ข้อมูลสังเคราะห์เพื่อการเรียนการสอน ไม่ใช่คำแนะนำทางคลินิก** ข้อความในคลังความรู้ คำถาม และคำสั่ง (prompt) ที่ส่งให้แบบจำลองใช้ภาษาอังกฤษ เนื่องจากแบบจำลองขนาดเล็กทำงานกับภาษาอังกฤษได้ดีที่สุด

In [ ]:
# @title ⚙️ การตั้งค่า — โปรดรันเซลล์นี้ก่อน (NAME และ JOIN_CODE ใช้เฉพาะโจทย์ท้าทายบนตารางอันดับ ซึ่งไม่บังคับ)
!pip -q install gradio_client huggingface_hub

HF_OWNER  = "wdittaya"     # บัญชี Hugging Face ที่ใช้โฮสต์ตารางอันดับของรายวิชา (ผู้สอนเป็นผู้กำหนด)
LAB_ID    = "lab09"
NAME      = ""            # ไม่บังคับ: ชื่อที่จะแสดงบนตารางอันดับแบบสาธารณะ (สามารถใช้นามแฝงได้)
JOIN_CODE = ""            # ไม่บังคับ: รหัสเข้าร่วมซึ่งผู้สอนประกาศในการเรียนการสอนผ่าน Zoom

SPACE_ID  = f"{HF_OWNER}/aidh-leaderboard"
DATA_REPO = f"{HF_OWNER}/aidh-lab-data"

from huggingface_hub import hf_hub_download

def lab_data(path):
    """ดาวน์โหลดไฟล์ข้อมูลทดสอบสาธารณะของรายวิชา 1 ไฟล์ (เช่น 'pima/test_ids.csv') และคืนค่าตำแหน่งไฟล์ในเครื่อง"""
    return hf_hub_download(DATA_REPO, path, repo_type="dataset")

def submit_to_leaderboard(path, lab=LAB_ID):
    """ส่งไฟล์ผลการทำนายไปยังตารางอันดับของรายวิชา และแสดงคะแนนพร้อมอันดับ"""
    from gradio_client import Client, handle_file
    assert NAME and JOIN_CODE, "โปรดระบุ NAME และ JOIN_CODE ในเซลล์การตั้งค่าก่อน (จำเป็นเฉพาะการส่งผลไปยังตารางอันดับ)"
    client = Client(SPACE_ID, verbose=False)
    print(client.predict(lab=lab, name=NAME, code=JOIN_CODE, file=handle_file(path), api_name="/submit"))

print(f"Leaderboard: https://huggingface.co/spaces/{SPACE_ID}  |  lab={LAB_ID}  name={NAME or '(ยังไม่ได้ระบุ — ตารางอันดับไม่บังคับ)'}")

## ความจำเป็นของ RAG

ปฏิบัติการที่ 8 แสดงให้เห็นว่า LLM อาจเกิด **hallucination** คือสร้างข้อความที่ดูน่าเชื่อถือแต่ไม่มีอยู่จริง **RAG (Retrieval-Augmented Generation)** ช่วยลดปัญหานี้ได้ส่วนหนึ่ง โดยก่อนที่ LLM จะตอบ ระบบจะ*ค้นคืน* (retrieve) ข้อความที่เกี่ยวข้องที่สุดจากคลังเอกสารที่เชื่อถือได้ (ในปฏิบัติการนี้คือแนวทางเวชปฏิบัติฉบับย่อ) แล้วนำไปใส่ใน prompt คำตอบที่ได้จึง **อ้างอิงเอกสาร (grounded)** และ **ตรวจสอบแหล่งที่มาได้ (citable)** ซึ่งเป็นคุณสมบัติสำคัญสำหรับการใช้งานทางคลินิก

ระบบประกอบด้วย 3 ส่วน ซึ่งเป็นโอเพนซอร์สและไม่มีค่าใช้จ่ายทั้งหมด ได้แก่

1. **Embedding** (`sentence-transformers`): แปลงแต่ละประโยคเป็นชุดตัวเลข โดยประโยคที่มี*ความหมาย*ใกล้เคียงกันจะได้ชุดตัวเลขที่ใกล้เคียงกัน แม้จะใช้คำต่างกัน เช่น "kidney failure" กับ "low eGFR"
2. **ฐานข้อมูลเวกเตอร์ (vector store)** (`faiss`): ดัชนีสำหรับค้นหาชุดตัวเลขดังกล่าวอย่างรวดเร็ว เทียบได้กับบัตรรายการของห้องสมุดที่ค้นหาข้อความซึ่งมีความหมายใกล้เคียงกับคำถามมากที่สุด
3. **LLM** (แบบจำลอง Qwen แบบโอเพนซอร์สจากปฏิบัติการที่ 8): เรียบเรียงคำตอบจากข้อความที่ค้นคืนได้เท่านั้น

เซลล์ถัดไปจะติดตั้งเครื่องมือและดาวน์โหลดแบบจำลอง embedding ขนาดเล็ก (ใช้เวลาประมาณ 1 นาที)

In [ ]:
!pip -q install sentence-transformers faiss-cpu "transformers>=4.44" accelerate
from sentence_transformers import SentenceTransformer
import faiss, numpy as np, torch
from transformers import pipeline

embedder = SentenceTransformer("all-MiniLM-L6-v2")   # แบบจำลอง embedding ขนาดเล็ก ไม่มีค่าใช้จ่าย และทำงานเร็ว
print("แบบจำลอง embedding พร้อมใช้งาน | ตรวจพบ GPU:", torch.cuda.is_available())

In [ ]:
# คลังความรู้ 'แนวทางเวชปฏิบัติ' ขนาดเล็ก (ข้อมูลสังเคราะห์เพื่อการเรียนการสอน ไม่ใช่คำแนะนำทางคลินิก)
kb = [
 "Type 2 diabetes is diagnosed when HbA1c is 6.5% or higher, or fasting plasma glucose is 126 mg/dL or higher.",
 "First-line pharmacotherapy for type 2 diabetes is metformin unless contraindicated.",
 "Blood pressure target for most adults with diabetes is below 130/80 mmHg.",
 "Statins are recommended for diabetic patients aged 40-75 to reduce cardiovascular risk.",
 "Metformin is contraindicated when eGFR is below 30 mL/min/1.73m2.",
 "Retinal (eye) screening is recommended at diagnosis of type 2 diabetes and then every 1-2 years.",
 "An HbA1c target below 7% is appropriate for most non-pregnant adults with diabetes; a less strict target (below 8%) may suit frail older adults.",
 "Mild hypoglycaemia (glucose below 70 mg/dL) in a conscious patient is treated with 15-20 g of fast-acting glucose, rechecking glucose after 15 minutes.",
]
emb = embedder.encode(kb, normalize_embeddings=True)   # เวกเตอร์ (ชุดตัวเลข) 1 ชุดต่อ 1 ข้อความ
index = faiss.IndexFlatIP(emb.shape[1])                  # ดัชนีสำหรับค้นหาด้วย cosine similarity
index.add(emb)
print("สร้างดัชนีแล้ว", index.ntotal, "ข้อความ แต่ละข้อความเป็นเวกเตอร์ขนาด", emb.shape[1], "มิติ")

def search(query, k=3):
    # คืนค่าข้อความที่คล้ายคลึงที่สุด k ข้อความ ในรูปคู่ (ข้อความ, คะแนนความคล้ายคลึง) เรียงจากมากไปน้อย
    q = embedder.encode([query], normalize_embeddings=True)
    scores, idx = index.search(q, k)          # คะแนนความคล้ายคลึง และลำดับที่ของข้อความ
    return [(kb[i], float(s)) for i, s in zip(idx[0], scores[0])]

def retrieve(query, k=3):
    # คืนค่าเฉพาะตัวข้อความ (ใช้ในฟังก์ชันตอบคำถามแบบ RAG ในขั้นถัดไป)
    return [p for p, s in search(query, k)]

💡 **คำอธิบายผลลัพธ์** ประโยคในแนวทางเวชปฏิบัติแต่ละประโยคถูกแปลงเป็นเวกเตอร์ (**embedding**) และจัดเก็บในดัชนี FAISS ฟังก์ชัน `search(question, k)` แปลงคำถามด้วยวิธีเดียวกัน แล้วคืนค่าข้อความ **top-k** (*k* ข้อความที่ใกล้เคียงที่สุด) พร้อม **คะแนนความคล้ายคลึง (similarity score)** โดยค่าประมาณ 1.0 หมายถึงความหมายเหมือนกัน และค่าประมาณ 0 หมายถึงไม่เกี่ยวข้องกัน ขั้นตอนนี้ยังไม่ใช้ LLM เป็นเพียงการค้นหาตามความหมาย (semantic search)

### แบบฝึกหัดที่ 1 (Exercise 1): การตั้งคำถามและการตรวจสอบข้อความที่ค้นคืนได้

In [ ]:
# ⭐ Exercise 1 — แบบฝึกหัดที่ 1: ตั้งคำถามและตรวจสอบข้อความที่ระบบค้นคืนได้
# 1) ให้แทน ... ด้วยคำถามทางคลินิก (ภาษาอังกฤษ) ในเครื่องหมายอัญประกาศ เช่น "Which diabetes drug should be avoided in kidney failure?"
# 2) ให้กำหนด TOP_K (จำนวนข้อความที่ต้องการค้นคืน) เป็นจำนวนเต็ม เช่น 3 แล้วทดลองใช้ 1 และ 5 เพื่อเปรียบเทียบ
MY_QUESTION = ...     # TODO: คำถามในเครื่องหมาย "อัญประกาศ"
TOP_K = ...           # TODO: จำนวนเต็ม เช่น 3

for rank, (passage, score) in enumerate(search(MY_QUESTION, TOP_K), start=1):
    print(f"#{rank}  ความคล้ายคลึง={score:.2f}  {passage}")

In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
MY_QUESTION = "Which diabetes drug should be avoided in kidney failure?"
TOP_K = 3

for rank, (passage, score) in enumerate(search(MY_QUESTION, TOP_K), start=1):
    print(f"#{rank}  ความคล้ายคลึง={score:.2f}  {passage}")

💡 **คำอธิบายผลลัพธ์** คำถามที่ใช้คำว่า "kidney failure" ไม่มีคำว่า *eGFR* ร่วมกับข้อความเรื่อง metformin แต่ embedding มักจัดให้ข้อความดังกล่าวอยู่ในอันดับต้น เนื่องจาก*ความหมาย*สัมพันธ์กัน ข้อความในอันดับถัดลงไปมีคะแนนความคล้ายคลึงต่ำกว่า และจะถูกส่งให้ LLM ด้วยหากกำหนด `TOP_K` สูงขึ้น

✍️ **คำถามที่ 1** ข้อความที่เป็นประโยชน์ที่สุดอยู่ในอันดับที่ 1 หรือไม่ เมื่อเพิ่มค่า `TOP_K` คะแนนความคล้ายคลึงของข้อความในอันดับท้ายเปลี่ยนแปลงอย่างไร และจงอธิบายว่าเหตุใดการส่งข้อความให้ LLM *มากเกินไป*จึงอาจก่อให้เกิดปัญหา

✍️ **คำตอบ:**

## การโหลดแบบจำลองภาษา (LLM)

ขั้นตอนนี้โหลด LLM ขนาดเล็กแบบโอเพนซอร์สตัวเดียวกับปฏิบัติการที่ 8 (Qwen2.5-1.5B-Instruct ขนาดประมาณ 3 GB ใช้เวลาดาวน์โหลด 1–2 นาที) ฟังก์ชัน `generate(system, user)` ส่งคำสั่ง **system** (ข้อกำหนดที่แบบจำลองต้องปฏิบัติตาม) และข้อความ **user** (คำถาม) ไปยังแบบจำลอง แล้วคืนค่าคำตอบของแบบจำลอง

In [ ]:
# LLM (แบบจำลองโอเพนซอร์สตัวเดียวกับปฏิบัติการที่ 8) ใช้ float16 บน GPU และ float32 บน CPU
llm = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct",
               torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
               device_map="auto")

def generate(system, user, max_new_tokens=250):
    msg = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    return llm(msg, max_new_tokens=max_new_tokens, do_sample=False)[0]["generated_text"][-1]["content"]

print("LLM พร้อมใช้งาน")

### แบบฝึกหัดที่ 2 (Exercise 2): prompt สำหรับ RAG

ฟังก์ชัน `answer_with_rag()` ค้นคืนข้อความที่เกี่ยวข้อง กำกับหมายเลข [1], [2], ... เป็น "บริบท (context)" แล้วสั่งให้ LLM ตอบ*จากบริบทเท่านั้น* พร้อมอ้างอิงหมายเลขข้อความ หรือตอบว่า `Not in the guidelines.` หากบริบทไม่มีคำตอบ คำสั่งที่ส่งให้แบบจำลองเขียนเป็นภาษาอังกฤษ เนื่องจากแบบจำลองขนาดเล็กปฏิบัติตามคำสั่งภาษาอังกฤษได้แม่นยำกว่า และต้องคงวลี `"Not in the guidelines."` ไว้ตามเดิม เนื่องจากระบบให้คะแนนใช้วลีนี้ในการตรวจสอบ

In [ ]:
# ⭐ Exercise 2 — แบบฝึกหัดที่ 2: prompt สำหรับ RAG
# ข้อความ user ที่ส่งให้แบบจำลองต้องประกอบด้วยทั้ง "บริบท" และ "คำถาม" โดยส่วนบริบทได้เตรียมไว้แล้ว
# ให้แทน ... ด้วยชื่อตัวแปรที่เก็บคำถาม   คำใบ้: ชื่อตัวแปรคือ  question
ABSTAIN_PHRASE = "Not in the guidelines."

def answer_with_rag(question, k=3, show_context=False):
    passages = retrieve(question, k)
    context = "\n".join(f"[{i+1}] {c}" for i, c in enumerate(passages))
    if show_context:
        print("บริบทที่ค้นคืนได้:\n" + context + "\n")
    system = ("You are a clinical guideline assistant. Answer ONLY using the "
              "provided context. If the answer is not in the context, reply exactly "
              f"'{ABSTAIN_PHRASE}' Cite passage numbers like [1].")
    user = f"Context:\n{context}\n\nQuestion: " + ...   # TODO: เพิ่มคำถาม
    return generate(system, user)

print(answer_with_rag("Can a patient with eGFR 25 take metformin?", show_context=True))

In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
ABSTAIN_PHRASE = "Not in the guidelines."

def answer_with_rag(question, k=3, show_context=False):
    passages = retrieve(question, k)
    context = "\n".join(f"[{i+1}] {c}" for i, c in enumerate(passages))
    if show_context:
        print("บริบทที่ค้นคืนได้:\n" + context + "\n")
    system = ("You are a clinical guideline assistant. Answer ONLY using the "
              "provided context. If the answer is not in the context, reply exactly "
              f"'{ABSTAIN_PHRASE}' Cite passage numbers like [1].")
    user = f"Context:\n{context}\n\nQuestion: " + question
    return generate(system, user)

print(answer_with_rag("Can a patient with eGFR 25 take metformin?", show_context=True))

💡 **คำอธิบายผลลัพธ์** ขั้นตอนนี้คือหลักการสำคัญของ RAG ได้แก่ *ค้นคืนข้อความ → นำข้อความใส่ใน prompt → สั่งให้ LLM ตอบจากข้อความดังกล่าวเท่านั้นพร้อมอ้างอิงแหล่งที่มา* นอกจากนี้ คำสั่ง system ยังกำหนดทางเลือกที่ปลอดภัยให้แบบจำลอง ("Not in the guidelines.") ซึ่งเรียกว่า **การงดตอบ (abstention)**

### แบบฝึกหัดที่ 3 (Exercise 3): การเปรียบเทียบคำตอบที่ไม่ใช้และใช้ RAG

In [ ]:
# ⭐ Exercise 3 — แบบฝึกหัดที่ 3: เปรียบเทียบคำตอบที่ไม่ค้นคืนเอกสารกับคำตอบที่ค้นคืนเอกสาร และประเมินคำตอบ
# ให้เลือกคำถาม 1 ข้อ โดยแทน ... ด้วยคำถามข้อใดข้อหนึ่งด้านล่าง (คัดลอกรวมเครื่องหมายอัญประกาศ) หรือตั้งคำถามเอง
#   "At what HbA1c is type 2 diabetes diagnosed?"
#   "What HbA1c target suits a frail 85-year-old with diabetes?"
#   "How should mild hypoglycaemia be treated in a conscious patient?"
COMPARE_QUESTION = ...   # TODO

print("คำถาม:", COMPARE_QUESTION)
print("\n--- A) ไม่ใช้ RAG (อาศัยความรู้ที่แบบจำลองเรียนรู้มา) ---\n" + generate("You are a clinical assistant.", COMPARE_QUESTION))
print("\n--- B) ใช้ RAG (อ้างอิงแนวทางเวชปฏิบัติ) ---")
print(answer_with_rag(COMPARE_QUESTION, show_context=True))

In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
COMPARE_QUESTION = "At what HbA1c is type 2 diabetes diagnosed?"

print("คำถาม:", COMPARE_QUESTION)
print("\n--- A) ไม่ใช้ RAG (อาศัยความรู้ที่แบบจำลองเรียนรู้มา) ---\n" + generate("You are a clinical assistant.", COMPARE_QUESTION))
print("\n--- B) ใช้ RAG (อ้างอิงแนวทางเวชปฏิบัติ) ---")
print(answer_with_rag(COMPARE_QUESTION, show_context=True))

💡 **คำอธิบายผลลัพธ์** คำตอบ A มาจากความรู้ที่แบบจำลองเรียนรู้ระหว่างการฝึกเท่านั้น จึงอาจถูกต้อง ล้าสมัย หรือไม่มีอยู่จริง และไม่สามารถตรวจสอบแหล่งที่มาได้ ส่วนคำตอบ B ควรอิงตามข้อความที่ค้นคืนได้และอ้างอิงหมายเลขข้อความ ซึ่งช่วยให้บุคลากรทางการแพทย์ตรวจสอบได้ภายในเวลาอันสั้น

✍️ **คำถามที่ 2 — การประเมินคำตอบในฐานะผู้ตรวจสอบทางคลินิก** ให้คัดลอกตารางต่อไปนี้ไปไว้ในเซลล์คำตอบและกรอกผล (ใช่ / ไม่ใช่ / บางส่วน)

| | ถูกต้องหรือไม่ | มีข้อความที่ค้นคืนได้รองรับหรือไม่ | อ้างอิงหมายเลขข้อความ [n] หรือไม่ | มีข้อมูลที่ไม่มีหลักฐานรองรับหรือไม่ปลอดภัยหรือไม่ |
|---|---|---|---|---|
| A) ไม่ใช้ RAG | | | | |
| B) ใช้ RAG | | | | |

จากผลลัพธ์ข้างต้น ผู้เรียนพิจารณาว่าคำตอบแบบใดเหมาะสมที่จะนำเสนอต่อแพทย์ผู้มีภาระงานมาก เพราะเหตุใด

✍️ **คำตอบ:**

## คำถามนอกขอบเขตและเกณฑ์การงดตอบ (abstention threshold)

ระบบผู้ช่วยทางคลินิกที่ปลอดภัยต้อง**งดตอบ**เมื่อแนวทางเวชปฏิบัติไม่ครอบคลุมคำถาม แทนการคาดเดา วิธีหนึ่งคือการใช้กฎอย่างง่าย*ก่อน*ส่งคำถามให้ LLM กล่าวคือ หากข้อความที่ค้นคืนได้ในอันดับที่ 1 มี **คะแนนความคล้ายคลึงต่ำกว่าเกณฑ์ตัดสิน (threshold)** ให้ตอบ "Not in the guidelines." ทันที

เซลล์ต่อไปนี้แสดงคะแนนความคล้ายคลึงสูงสุดของชุดคำถามฝึกหัด ซึ่งมีทั้งคำถามที่แนวทางเวชปฏิบัติฉบับย่อครอบคลุม (**อยู่ในขอบเขต / in scope**) และคำถามที่ไม่ครอบคลุม (**นอกขอบเขต / out of scope**) ขั้นตอนนี้ไม่ใช้ LLM จึงทำงานได้รวดเร็ว

In [ ]:
practice = [   # (คำถาม, แนวทางเวชปฏิบัติฉบับย่อครอบคลุมหรือไม่)
    ("At what HbA1c level is type 2 diabetes diagnosed?",               True),
    ("Can a patient with an eGFR of 25 keep taking metformin?",         True),
    ("How often should people with diabetes have their eyes checked?",  True),
    ("What blood pressure should a diabetic patient aim for?",          True),
    ("What is the first-line treatment for head lice in children?",     False),
    ("How should a sprained ankle be managed in the first 48 hours?",  False),
    ("Which vaccines are due at a baby's 2-month visit?",              False),
    ("What is the first-line antidepressant for major depression?",     False),
]

def best_score(question):
    return search(question, 1)[0][1]      # คะแนนความคล้ายคลึงของข้อความอันดับที่ 1

print(f"{'คะแนนสูงสุด':>10}  {'ในขอบเขต':9}  คำถาม")
for qtext, in_scope in practice:
    print(f"{best_score(qtext):10.2f}  {'ใช่' if in_scope else 'ไม่ใช่':9}  {qtext}")

### แบบฝึกหัดที่ 4 (Exercise 4): การกำหนดเกณฑ์การงดตอบ

In [ ]:
# ⭐ Exercise 4 — แบบฝึกหัดที่ 4: กำหนดเกณฑ์การงดตอบ (abstention threshold)
# ให้พิจารณาตารางข้างต้น แล้วเลือกค่าที่อยู่ "ระหว่าง" คะแนนของคำถามในขอบเขต (ซึ่งมักสูงกว่า)
# และคะแนนของคำถามนอกขอบเขต (ซึ่งมักต่ำกว่า) คำถามที่มีคะแนนสูงสุดต่ำกว่าเกณฑ์นี้จะถูกงดตอบ
# คำใบ้: เริ่มจาก 0.4 รันเซลล์และพิจารณาผลลัพธ์ จากนั้นปรับค่าให้สูงขึ้นหรือต่ำลง
ABSTAIN_BELOW = ...    # TODO: ค่าระหว่าง 0 ถึง 1 เช่น 0.4

correct = 0
for qtext, in_scope in practice:
    refuse = best_score(qtext) < ABSTAIN_BELOW
    ok = (refuse != in_scope)             # งดตอบเฉพาะคำถามนอกขอบเขต
    correct += ok
    print(f"{'งดตอบ' if refuse else 'ตอบ':6}  {'ถูก ' if ok else 'ผิด '}  {qtext}")
print(f"\nความแม่นยำของการงดตอบในชุดคำถามฝึกหัด: {correct}/{len(practice)}")

In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
# เลือกค่าโดยอัตโนมัติ: ทดสอบเกณฑ์ที่อยู่กึ่งกลางระหว่างคะแนนที่อยู่ติดกัน แล้วเลือกค่าที่ให้จำนวนข้อถูกมากที่สุด
# (หากเท่ากัน ให้เลือกเกณฑ์ที่ต่ำกว่า ซึ่งงดตอบน้อยกว่า)
scored = sorted((best_score(q), s) for q, s in practice)
candidates = [0.0] + [round((a[0] + b[0]) / 2, 3) for a, b in zip(scored, scored[1:])]
def n_right(t):
    return sum((sc < t) != s for sc, s in scored)
ABSTAIN_BELOW = max(candidates, key=lambda t: (n_right(t), -t))
print("ABSTAIN_BELOW =", ABSTAIN_BELOW)

correct = 0
for qtext, in_scope in practice:
    refuse = best_score(qtext) < ABSTAIN_BELOW
    ok = (refuse != in_scope)
    correct += ok
    print(f"{'งดตอบ' if refuse else 'ตอบ':6}  {'ถูก ' if ok else 'ผิด '}  {qtext}")
print(f"\nความแม่นยำของการงดตอบในชุดคำถามฝึกหัด: {correct}/{len(practice)}")

💡 **คำอธิบายผลลัพธ์** ขั้นตอนนี้เป็นการกำหนดกลไกป้องกันด้านความปลอดภัย (guard-rail) เกณฑ์ที่**สูงเกินไป**จะทำให้ระบบงดตอบคำถามที่แนวทางเวชปฏิบัติ*มี*คำตอบ (ไม่สะดวกต่อผู้ใช้ แต่ปลอดภัย) ส่วนเกณฑ์ที่**ต่ำเกินไป**จะทำให้คำถามที่ไม่เกี่ยวข้องผ่านไปถึง LLM ซึ่งอาจสร้างคำตอบที่ไม่มีหลักฐานรองรับ (มีความเสี่ยง) เซลล์ถัดไปรวมกลไกป้องกันนี้เข้ากับ LLM

In [ ]:
def answer_safely(question, k=3):
    # ตรวจสอบเกณฑ์ก่อน (ไม่ใช้ LLM) แล้วจึงตอบแบบ RAG ที่อ้างอิงเอกสาร (LLM ยังสามารถงดตอบเองได้)
    if best_score(question) < ABSTAIN_BELOW:
        return ABSTAIN_PHRASE + " (คะแนนความคล้ายคลึงสูงสุดต่ำกว่าเกณฑ์)"
    return answer_with_rag(question, k)

# ทดสอบคำถามในขอบเขต 1 ข้อและนอกขอบเขต 1 ข้อ (ผู้เรียนสามารถเปลี่ยนเป็นคำถามของตนเองได้)
for qtext in ["Should a 60-year-old with diabetes take a statin?",
              "How is scabies treated?"]:
    print("คำถาม:", qtext, "\nคำตอบ:", answer_safely(qtext), "\n")

✍️ **คำถามที่ 3** โปรดระบุเกณฑ์ที่เลือกและจำนวนคำถามฝึกหัดที่ระบบตัดสินได้ถูกต้อง จากนั้นจงอธิบายใน 2–3 ประโยคว่า ในมุมมองความปลอดภัยของผู้ป่วย ความผิดพลาดประเภทใดร้ายแรงกว่าสำหรับระบบผู้ช่วยด้านแนวทางเวชปฏิบัติ ระหว่างการงดตอบคำถามที่ตอบได้ กับการตอบคำถามที่อยู่นอกแนวทางเวชปฏิบัติ เพราะเหตุใด

✍️ **คำตอบ:**

## จาก RAG สู่เอเจนต์ปัญญาประดิษฐ์ (AI Agent)

**เอเจนต์ปัญญาประดิษฐ์ (AI agent)** สามารถ*ตัดสินใจ*เลือกใช้ **เครื่องมือ (tool)** เช่น การค้นแนวทางเวชปฏิบัติ เครื่องคำนวณ การตรวจสอบอันตรกิริยาระหว่างยา (drug interaction) หรือการดึงข้อมูลจากระบบเวชระเบียนอิเล็กทรอนิกส์ (EHR) แล้วดำเนินการตามผลลัพธ์ ซึ่งมักทำซ้ำหลายรอบ เซลล์ด้านล่างเป็นตัวอย่างการเลือกเครื่องมืออย่างง่ายโดยไม่ใช้ framework เพื่อแสดงหลักการ ประกอบด้วยเครื่องมือคำนวณ BMI และ RAG แบบมีกลไกป้องกันเป็นทางเลือกสำรอง

In [ ]:
import re

def bmi_tool(weight_kg, height_m):
    return round(weight_kg / (height_m ** 2), 1)

def simple_agent(question):
    # เลือกเครื่องมือด้วยคำสำคัญอย่างง่าย ส่วนเอเจนต์จริงจะให้ LLM สร้างคำสั่งเรียกเครื่องมือแบบมีโครงสร้าง (เช่น JSON)
    if "bmi" in question.lower():
        nums = [float(x) for x in re.findall(r"\d+(?:\.\d+)?", question)]
        if len(nums) >= 2:
            return f"[เครื่องมือ: คำนวณ BMI] BMI = {bmi_tool(nums[0], nums[1])} kg/m2"
    return "[เครื่องมือ: RAG แนวทางเวชปฏิบัติ] " + answer_safely(question)

print(simple_agent("Compute BMI for weight 82 kg and height 1.75 m."))
print(simple_agent("Should diabetic patients aged 40-75 take statins?"))

💡 **คำอธิบายผลลัพธ์** เอเจนต์พิจารณาคำขอ แล้วเลือกเครื่องคำนวณสำหรับคำถามเรื่อง BMI (ให้ผลการคำนวณที่แม่นยำ ขณะที่ LLM มักคำนวณผิดพลาด) และเลือก RAG แนวทางเวชปฏิบัติสำหรับคำถามทางคลินิก framework สำหรับเอเจนต์ที่ใช้งานจริงทำงานในลักษณะเดียวกัน แต่ให้ LLM เป็นผู้เลือกเครื่องมือและข้อมูลนำเข้า ซึ่งยืดหยุ่นกว่า ทั้งนี้จึงจำเป็นต้องบันทึกและตรวจสอบการดำเนินการของเอเจนต์ทุกครั้ง

✍️ **คำถามที่ 4** โปรดระบุเครื่องมือ 1 ประเภทที่ผู้เรียนเห็นว่าควรให้เอเจนต์ปัญญาประดิษฐ์ทางคลินิกในหน่วยงานของตนใช้ได้ (เช่น การตรวจสอบอันตรกิริยาระหว่างยา การเรียกดูผลการตรวจทางห้องปฏิบัติการ การนัดหมาย) และความเสี่ยงด้านความปลอดภัย 1 ประการ หากให้เอเจนต์ใช้เครื่องมือนั้นโดยอัตโนมัติ

✍️ **คำตอบ:**

✍️ **คำถามที่ 5 (สรุป)** จงอธิบายใน 2–3 ประโยคว่า การให้ระบบปัญญาประดิษฐ์ตอบโดยอ้างอิงแนวทางเวชปฏิบัติที่ค้นคืนได้ (พร้อมการอ้างอิงแหล่งที่มาและการงดตอบ) ส่งผลต่อความน่าเชื่อถือของเครื่องมือดังกล่าวในเวชปฏิบัติอย่างไร และประเด็นใดที่ผู้ใช้ยังต้องตรวจสอบด้วยตนเอง

✍️ **คำตอบ:**

## 🏆 โจทย์ท้าทายบนตารางอันดับ (Leaderboard) — ไม่บังคับ สำหรับผู้เรียนที่เขียนโปรแกรมได้

> **ส่วนนี้ไม่บังคับ และไม่นับเป็นส่วนหนึ่งของชิ้นงานที่ต้องส่ง** ผู้เรียนสามารถทำปฏิบัติการให้เสร็จสมบูรณ์โดยไม่ต้องทำส่วนนี้ โค้ดตั้งต้น (baseline) ด้านล่างสามารถรันได้ทันทีและสร้างไฟล์ผลลัพธ์ในรูปแบบที่ถูกต้อง ผู้เรียนทุกคนจึง*สามารถ*ส่งผลได้ ส่วนผู้เรียนที่เขียนโปรแกรมได้สามารถพัฒนาระบบให้ได้คะแนนสูงกว่า baseline

รายวิชาจัดเตรียม **คลังความรู้แนวทางเวชปฏิบัติสังเคราะห์ 40 ข้อความ** (`rag/kb.json`, รหัส `P01`–`P40`) และ **คำถาม 22 ข้อ** (`rag/questions.json`) ซึ่ง**บางข้อ*ไม่สามารถ*ตอบได้จากคลังความรู้นี้** สำหรับคำถามแต่ละข้อ ให้ส่งรหัสข้อความ 3 อันดับแรกที่ค้นคืนได้และคำตอบที่ระบบสร้างขึ้น ในไฟล์ `lab09_submission.json` ตามรูปแบบต่อไปนี้

```json
{"Q01": {"passages": ["P12", "P03", "P30"], "answer": "..."}, ...}
```

**ตัวชี้วัด: 0.5 × Recall@3** (สำหรับคำถามในขอบเขต: มีข้อความที่ถูกต้องอยู่ใน 3 อันดับแรกหรือไม่) **+ 0.5 × ความแม่นยำของการงดตอบ (abstention accuracy)** (คำตอบมีวลี *"Not in the guidelines"* เฉพาะเมื่อคำถามอยู่นอกขอบเขตหรือไม่)

หากต้องการส่งผล ให้ระบุ `NAME` และ `JOIN_CODE` ในเซลล์ ⚙️ การตั้งค่า ด้านบน แล้วรันเซลล์นั้นอีกครั้ง เซลล์ baseline จะ**แทนที่** `kb` และ `index` ขนาดเล็กที่ใช้ก่อนหน้า ดังนั้น `search()`, `answer_with_rag()` และ `answer_safely()` จะทำงานบนคลังความรู้ใหม่โดยอัตโนมัติ

**หลักความซื่อสัตย์ทางวิชาการ (honor code):** ผลที่ส่งต้องได้จากระบบ (pipeline) ของผู้เรียนเอง ห้ามกำหนดป้ายกำกับของคำถามทดสอบทั้ง 22 ข้อด้วยตนเอง และห้ามเลือกรหัสข้อความหรือคำตอบให้คำถามเหล่านั้นด้วยตนเอง

In [ ]:
# 🏆 Baseline — ค้นคืนข้อความและตอบคำถามของรายวิชา 22 ข้อ (รันได้ทันที ใช้เวลาประมาณ 2-4 นาทีบน T4 GPU)
import json
kb_docs   = json.load(open(lab_data("rag/kb.json"), encoding="utf-8"))
questions = json.load(open(lab_data("rag/questions.json"), encoding="utf-8"))
kb_ids = [d["id"] for d in kb_docs]
kb = [d["text"] for d in kb_docs]                        # แทนที่คลังความรู้ขนาดเล็กก่อนหน้า
emb = embedder.encode(kb, normalize_embeddings=True)
index = faiss.IndexFlatIP(emb.shape[1]); index.add(emb)

LB_ABSTAIN_BELOW = 0.35   # เกณฑ์ความคล้ายคลึงสำหรับคลังความรู้นี้ (ค่าจากส่วนหลักอาจใช้กับคลังนี้ไม่ได้) ปรับค่าได้ตามต้องการ
USE_LLM = True            # False = ไม่ใช้ LLM (เร็วกว่ามาก เนื่องจากคะแนนพิจารณาเฉพาะการตัดสินใจงดตอบ)

submission = {}
for q in questions:
    qv = embedder.encode([q["question"]], normalize_embeddings=True)
    scores, idx = index.search(qv, 3)
    top = [kb_ids[i] for i in idx[0]]
    if scores[0][0] < LB_ABSTAIN_BELOW:
        answer = ABSTAIN_PHRASE
    elif USE_LLM:
        answer = answer_with_rag(q["question"])           # LLM อาจตัดสินใจตอบ 'Not in the guidelines' เอง
    else:
        answer = "See passages " + ", ".join(top)
    submission[q["id"]] = {"passages": top, "answer": answer}
    print(q["id"], round(float(scores[0][0]), 2), answer[:80].replace("\n", " "))

json.dump(submission, open("lab09_submission.json", "w", encoding="utf-8"), indent=1)
print("\nบันทึกไฟล์ lab09_submission.json แล้ว จำนวนคำตอบ", len(submission), "ข้อ")
if NAME and JOIN_CODE:
    submit_to_leaderboard("lab09_submission.json")
else:
    print("โปรดระบุ NAME และ JOIN_CODE ในเซลล์การตั้งค่า (และรันเซลล์นั้นอีกครั้ง) เพื่อส่งผลไปยังตารางอันดับ")

**แนวทางการพัฒนาให้ได้คะแนนสูงกว่า baseline**

- **ปรับเกณฑ์การงดตอบ** — แสดงคะแนนสูงสุดของคำถามแต่ละข้อ แล้วพิจารณาการกระจายของคะแนน (โดยไม่กำหนดป้ายกำกับของคำถามทดสอบด้วยตนเอง)
- **ใช้ embedding ที่มีประสิทธิภาพสูงกว่า** — เช่น `BAAI/bge-small-en-v1.5` หรือ `intfloat/e5-small-v2` (โปรดศึกษา model card เนื่องจากบางแบบจำลองกำหนดให้เติมคำนำหน้าคำถาม)
- **การค้นคืนแบบผสม (hybrid retrieval)** — รวมคะแนนคำสำคัญแบบ BM25 (`pip install rank_bm25`) เข้ากับคะแนนความคล้ายคลึงแบบ dense
- **การจัดอันดับใหม่ (re-ranking)** — ให้คะแนนข้อความ 10 อันดับแรกใหม่ด้วย cross-encoder เช่น `cross-encoder/ms-marco-MiniLM-L-6-v2`
- **ให้ LLM ประเมินความเกี่ยวข้อง** — ใช้ prompt ที่เข้มงวดขึ้น หรือถามแบบจำลองก่อนว่า "Is the answer contained in the context? yes/no" แล้วจึงให้ตอบ
- คำตอบของคำถามนอกขอบเขตต้องมีวลี *"Not in the guidelines"* ตรงตามที่กำหนด

In [ ]:
# (ไม่บังคับ) ระบบที่ผู้เรียนพัฒนาเอง: สร้างไฟล์ lab09_submission.json ใหม่ในรูปแบบเดียวกัน แล้วส่งผล
# submit_to_leaderboard("lab09_submission.json")

## ชิ้นงานที่ต้องส่ง (งานเดี่ยว)

ให้ส่ง**โน้ตบุ๊กนี้ที่รันครบทุกเซลล์ตามลำดับจากบนลงล่าง** (รันเซลล์ส่วนหลัก ✅ ครบถ้วน) พร้อมคำตอบของคำถาม ✍️ ทั้ง 5 ข้อ โน้ตบุ๊กควรแสดงข้อความที่ค้นคืนได้สำหรับคำถามของผู้เรียน การเปรียบเทียบคำตอบที่ไม่ใช้และใช้ RAG พร้อมผลการประเมิน เกณฑ์การงดตอบที่เลือก และการงดตอบคำถามนอกขอบเขตได้อย่างถูกต้อง โดยส่งตามช่องทางที่ระบุในเอกสารรายวิชา

ผู้เรียนที่ทำโจทย์ท้าทาย 🏆 บนตารางอันดับ สามารถระบุชื่อบนตารางอันดับและคะแนนสูงสุดไว้ในส่วนนี้ (ไม่บังคับ)

✍️ **ชื่อและคะแนนบนตารางอันดับ (ไม่บังคับ):**

## แหล่งศึกษาเพิ่มเติม
เครื่องมือสร้างระบบ RAG แบบไม่ต้องเขียนโปรแกรมหรือเขียนโปรแกรมเพียงเล็กน้อย (no-code/low-code) ได้แก่ **[Dify](https://dify.ai/)**, **[Langflow](https://www.langflow.org/)** และ **[Flowise](https://flowiseai.com/)** ซึ่งใช้แนวคิดเดียวกัน (embedding, vector store, การค้นคืน และ prompt) ผ่านการลากและวางองค์ประกอบ